In [1]:
import pandas as pd

# loading the raw CSV
df = pd.read_csv('kaggle raw data/games.csv')

# previewing the data
print(df.shape)
df.head()

(74026, 23)


,game_id,competition_id,season,round,date,home_club_id,away_club_id,home_club_goals,away_club_goals,home_club_position,...,stadium,attendance,referee,url,home_club_formation,away_club_formation,home_club_name,away_club_name,aggregate,competition_type
0,2321027,L1,2013,1. Matchday,2013-08-11,33.0,41.0,3.0,3.0,8.0,...,Veltins-Arena,61973.0,Manuel Gräfe,https://www.transfermarkt.co.uk/fc-schalke-04_...,4-2-3-1,4-2-3-1,FC Schalke 04,Hamburger SV,3:3,domestic_league
1,2321033,L1,2013,1. Matchday,2013-08-10,23.0,86.0,0.0,1.0,13.0,...,EINTRACHT-Stadion,23000.0,Deniz Aytekin,https://www.transfermarkt.co.uk/eintracht-brau...,4-3-2-1,4-3-1-2,Eintracht Braunschweig,Sportverein Werder Bremen von 1899,0:1,domestic_league
2,2321044,L1,2013,2. Matchday,2013-08-18,16.0,23.0,2.0,1.0,1.0,...,SIGNAL IDUNA PARK,80200.0,Peter Sippel,https://www.transfermarkt.co.uk/borussia-dortm...,4-2-3-1,4-3-2-1,Borussia Dortmund,Eintracht Braunschweig,2:1,domestic_league
3,2321060,L1,2013,3. Matchday,2013-08-25,23.0,24.0,0.0,2.0,18.0,...,EINTRACHT-Stadion,23325.0,Wolfgang Stark,https://www.transfermarkt.co.uk/eintracht-brau...,4-3-2-1,4-2-3-1,Eintracht Braunschweig,Eintracht Frankfurt Fußball AG,0:2,domestic_league
4,2321072,L1,2013,5. Matchday,2013-09-14,16.0,41.0,6.0,2.0,1.0,...,SIGNAL IDUNA PARK,80645.0,Tobias Welz,https://www.transfermarkt.co.uk/borussia-dortm...,4-2-3-1,3-5-2,Borussia Dortmund,Hamburger SV,6:2,domestic_league


In [2]:
# checking for missing values
df.isna().sum().sort_values(ascending=False)

home_club_position        22467
away_club_position        22467
home_club_name            12850
away_club_name            11455
attendance                 9948
home_club_formation        6975
away_club_formation        6806
home_club_manager_name      828
away_club_manager_name      828
referee                     652
stadium                     250
aggregate                    12
home_club_goals              12
away_club_goals              12
home_club_id                  9
away_club_id                  9
game_id                       0
url                           0
competition_id                0
date                          0
round                         0
season                        0
competition_type              0
dtype: int64

In [3]:
df = df.drop(columns=['home_club_position', 'away_club_position'])

In [4]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

In [5]:
# filling with placeholder
df['referee'] = df['referee'].fillna('Unknown')
df['home_club_manager_name'] = df['home_club_manager_name'].fillna('Unknown')
df['attendance'] = df['attendance'].fillna(0)

In [6]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

In [7]:
df = df.drop_duplicates()

In [8]:
df['goal_diff'] = df['home_club_goals'] - df['away_club_goals']

In [9]:
df = df.dropna(subset=[
    'home_club_name', 'away_club_name',
    'home_club_formation', 'away_club_formation',
    'stadium'
])

In [10]:
print(df.dtypes)

game_id                            int64
competition_id                    object
season                             int64
round                             object
date                      datetime64[ns]
home_club_id                     float64
away_club_id                     float64
home_club_goals                  float64
away_club_goals                  float64
home_club_manager_name            object
away_club_manager_name            object
stadium                           object
attendance                       float64
referee                           object
url                               object
home_club_formation               object
away_club_formation               object
home_club_name                    object
away_club_name                    object
aggregate                         object
competition_type                  object
goal_diff                        float64
dtype: object


In [11]:
df['home_club_id'] = df['home_club_id'].astype(int)
df['away_club_id'] = df['away_club_id'].astype(int)

In [12]:
df.reset_index(drop=True, inplace=True)

In [14]:
df['total_goals'] = df['home_club_goals'] + df['away_club_goals']
df['is_draw'] = df['home_club_goals'] == df['away_club_goals']
df['is_home_win'] = df['home_club_goals'] > df['away_club_goals']

In [16]:
df.head()

,game_id,competition_id,season,round,date,home_club_id,away_club_id,home_club_goals,away_club_goals,home_club_manager_name,...,home_club_formation,away_club_formation,home_club_name,away_club_name,aggregate,competition_type,goal_diff,total_goals,is_draw,is_home_win
0,2321027,L1,2013,1. Matchday,2013-08-11,33,41,3.0,3.0,Jens Keller,...,4-2-3-1,4-2-3-1,FC Schalke 04,Hamburger SV,3:3,domestic_league,0.0,6.0,True,False
1,2321033,L1,2013,1. Matchday,2013-08-10,23,86,0.0,1.0,Torsten Lieberknecht,...,4-3-2-1,4-3-1-2,Eintracht Braunschweig,Sportverein Werder Bremen von 1899,0:1,domestic_league,-1.0,1.0,False,False
2,2321044,L1,2013,2. Matchday,2013-08-18,16,23,2.0,1.0,Jürgen Klopp,...,4-2-3-1,4-3-2-1,Borussia Dortmund,Eintracht Braunschweig,2:1,domestic_league,1.0,3.0,False,True
3,2321060,L1,2013,3. Matchday,2013-08-25,23,24,0.0,2.0,Torsten Lieberknecht,...,4-3-2-1,4-2-3-1,Eintracht Braunschweig,Eintracht Frankfurt Fußball AG,0:2,domestic_league,-2.0,2.0,False,False
4,2321072,L1,2013,5. Matchday,2013-09-14,16,41,6.0,2.0,Jürgen Klopp,...,4-2-3-1,3-5-2,Borussia Dortmund,Hamburger SV,6:2,domestic_league,4.0,8.0,False,True


In [17]:
print(df.isna().sum())

game_id                    0
competition_id             0
season                     0
round                      0
date                       0
home_club_id               0
away_club_id               0
home_club_goals            0
away_club_goals            0
home_club_manager_name     0
away_club_manager_name    58
stadium                    0
attendance                 0
referee                    0
url                        0
home_club_formation        0
away_club_formation        0
home_club_name             0
away_club_name             0
aggregate                  0
competition_type           0
goal_diff                  0
total_goals                0
is_draw                    0
is_home_win                0
dtype: int64


In [18]:
df['away_club_manager_name'] = df['away_club_manager_name'].fillna('Unknown')

In [19]:
print(df.isna().sum())

game_id                   0
competition_id            0
season                    0
round                     0
date                      0
home_club_id              0
away_club_id              0
home_club_goals           0
away_club_goals           0
home_club_manager_name    0
away_club_manager_name    0
stadium                   0
attendance                0
referee                   0
url                       0
home_club_formation       0
away_club_formation       0
home_club_name            0
away_club_name            0
aggregate                 0
competition_type          0
goal_diff                 0
total_goals               0
is_draw                   0
is_home_win               0
dtype: int64


In [20]:
df.to_csv('cleaned_games.csv', index=False)